# KIS — blind VLM benchmark trên Kaggle
Chọn **GPU T4** và bật **Internet**. Thêm ZIP dưới dạng **private Dataset**. Không cần khóa Drive/API.
Không chứa ground truth; kết quả được đối chiếu nhãn trên máy sau khi tải về.

In [ ]:
from pathlib import Path
import zipfile, sys, shutil
input_root = Path('/kaggle/input')
inputs = list(input_root.rglob('kis-vlm-benchmark.zip'))
work = Path('/kaggle/working/benchmark_code')
work.mkdir(exist_ok=True)
root = work / 'kis_vlm_benchmark'
if inputs:
    assert len(inputs) == 1, 'Add only one benchmark ZIP'
    with zipfile.ZipFile(inputs[0]) as z:
        for name in z.namelist():
            p = Path(name)
            assert not p.is_absolute() and '..' not in p.parts, 'Unsafe archive path'
        z.extractall(work)
else:
    # Kaggle may expand uploaded ZIPs automatically. Accept the same allowlisted contents.
    manifests = [p for p in input_root.rglob('manifest.json') if p.parent.name == 'vlm_packet' and (p.parent.parent / 'tools/kaggle_vlm_benchmark.py').is_file()]
    assert len(manifests) == 1, 'Add the private benchmark Dataset (ZIP or expanded contents)'
    shutil.copytree(manifests[0].parent.parent, root, dirs_exist_ok=True)
sys.path.insert(0, str(root))
print(root)


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'config/requirements-kaggle-vlm.txt')], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU in Notebook Settings'
print(torch.cuda.get_device_name(0))
from tools.kaggle_vlm_benchmark import run
# Use 100 for an initial smoke run; then None to continue all remaining tasks.
# Repeating this cell resumes completed task IDs without changing model/prompt/settings.
MAX_TASKS = None
OUTPUT = Path('/kaggle/working/vlm-results')
run(root / 'vlm_packet', OUTPUT, max_tasks=MAX_TASKS)


In [ ]:
import shutil
from IPython.display import FileLink, display
path = shutil.make_archive('/kaggle/working/vlm-results', 'zip', OUTPUT)
display(FileLink(path))
